# AnyUp Sub-Patch Information — MVP Setup

This notebook is a shared starting point for the group. It covers:
- **Setup**: install deps, load encoders (DINOv2, CLIP, SigLIP), load AnyUp
- **Token extraction**: pull patch tokens from a batch of ImageNet images
- **MLP probe**: train a small decoder to reconstruct the original patch from its token
- **E3 sketch**: compare reconstruction quality from raw token vs AnyUp-upsampled feature

Everything runs on CPU for exploration; swap `device = 'cuda'` when on the UdS cluster.

---

## 0 — Install dependencies

In [1]:
# Run once; restart kernel after.
# On the cluster, use uv pip as in the AnyUp README.
!pip install -q torch torchvision open_clip_torch einops matplotlib scikit-image
# SigLIP is bundled inside open_clip_torch (>=2.24)
print('Done')

Done


## 1 — Config

In [2]:
import torch
import torchvision.transforms as T
from pathlib import Path

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
PATCH_SIZE  = 14          # ViT-B/14 (DINOv2, SigLIP); CLIP uses 16 — see loader below
IMG_SIZE    = 224
BATCH_SIZE  = 32          # lower if you hit OOM
CACHE_DIR   = Path('token_cache')   # cached features land here
CACHE_DIR.mkdir(exist_ok=True)

# ImageNet val — update this path on your machine
IMAGENET_VAL = Path('/path/to/imagenet/val')

print(f'Device: {DEVICE}')

Device: cpu


## 2 — ImageNet-style transform

In [ ]:
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std  = (0.229, 0.224, 0.225)

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# Quick smoke-test with a single random tensor
dummy = transform(T.ToPILImage()(torch.rand(3, 256, 256)))
print('Transformed shape:', dummy.shape)

## 3 — Load encoders

Each returns a `(B, N_patches, C)` tensor of patch tokens.
We wrap them in a common interface so the rest of the notebook is encoder-agnostic.

In [ ]:
import open_clip

# ── DINOv2 ──────────────────────────────────────────────────────────────────
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', pretrained=True)
dino = dino.eval().to(DEVICE)

@torch.no_grad()
def encode_dino(imgs, layers=None):
    """
    imgs : (B,3,H,W) normalised tensor
    layers : list of ints (0-indexed) to return intermediate features;
             None → return only the final layer
    Returns dict {layer_idx: (B, N, C)} where layer_idx=-1 is the last layer.
    """
    if layers is None:
        out = dino.forward_features(imgs.to(DEVICE))
        return {-1: out['x_norm_patchtokens']}   # (B, N, C)
    else:
        # Register hooks for intermediate layers
        intermediates = {}
        hooks = []
        for l in layers:
            def make_hook(layer_idx):
                def hook(module, inp, outp):
                    # outp is (B, N+1, C); drop CLS
                    intermediates[layer_idx] = outp[:, 1:, :].detach().cpu()
                return hook
            hooks.append(dino.blocks[l].register_forward_hook(make_hook(l)))
        dino.forward_features(imgs.to(DEVICE))
        for h in hooks:
            h.remove()
        return intermediates

print('DINOv2 loaded ✓')

In [ ]:
# ── CLIP (ViT-B/16) ──────────────────────────────────────────────────────────
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16', pretrained='openai'
)
clip_model = clip_model.visual.eval().to(DEVICE)
CLIP_PATCH_SIZE = 16

@torch.no_grad()
def encode_clip(imgs):
    """
    Returns {-1: (B, N, C)} patch tokens (CLS dropped) from the final layer.
    """
    # open_clip visual forward returns (B, N+1, C) when output_tokens=True
    tokens = clip_model(imgs.to(DEVICE), output_tokens=True)   # (B, N+1, C)
    return {-1: tokens[:, 1:, :]}   # drop CLS

print('CLIP loaded ✓')

In [ ]:
# ── SigLIP (ViT-B/16-SigLIP, available through open_clip) ───────────────────
siglip_model, _, siglip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16-SigLIP', pretrained='webli'
)
siglip_model = siglip_model.visual.eval().to(DEVICE)

@torch.no_grad()
def encode_siglip(imgs):
    tokens = siglip_model(imgs.to(DEVICE), output_tokens=True)
    return {-1: tokens[:, 1:, :]}

print('SigLIP loaded ✓')

## 4 — Load AnyUp

In [ ]:
# Multi-backbone version generalises to any encoder — use this for E3
anyup = torch.hub.load('wimmerth/anyup', 'anyup_multi_backbone', use_natten=False)
anyup = anyup.eval().to(DEVICE)
print('AnyUp loaded ✓')

@torch.no_grad()
def upsample_features(hr_image, lr_features_grid):
    """
    hr_image         : (B, 3, H, W) ImageNet-normalised
    lr_features_grid : (B, C, h, w) patch features as a spatial grid
    Returns          : (B, C, H, W) upsampled feature map
    """
    return anyup(hr_image.to(DEVICE), lr_features_grid.to(DEVICE))

## 5 — Patch extraction utilities

Given a batch of images and their patch tokens, extract the **raw pixel patches** (the ground-truth targets for the MLP probe).

In [ ]:
import torch.nn.functional as F

def tokens_to_grid(tokens, h, w):
    """Reshape (B, N, C) → (B, C, h, w)."""
    B, N, C = tokens.shape
    assert N == h * w
    return tokens.permute(0, 2, 1).reshape(B, C, h, w)

def extract_pixel_patches(imgs_unnorm, patch_size):
    """
    imgs_unnorm : (B, 3, H, W)  — UNnormalised, values in [0,1]
    Returns     : (B*N, 3, P, P) one patch per row, matching token order
    """
    B, C, H, W = imgs_unnorm.shape
    P = patch_size
    # unfold into patches
    patches = imgs_unnorm.unfold(2, P, P).unfold(3, P, P)   # (B,3,h,w,P,P)
    patches = patches.contiguous().view(B, C, -1, P, P)      # (B,3,N,P,P)
    patches = patches.permute(0, 2, 1, 3, 4)                 # (B,N,3,P,P)
    return patches.reshape(-1, C, P, P)                       # (B*N,3,P,P)

# Quick shape check
dummy_imgs = torch.rand(2, 3, 224, 224)
patches = extract_pixel_patches(dummy_imgs, patch_size=14)
print('Patch tensor shape (B*N, 3, P, P):', patches.shape)   # expect (2*256, 3, 14, 14)

## 6 — MLP probe (shared across E1, E2, E3)

Exactly the architecture from the paper: `Linear(C,1024) → GELU → Linear(1024,1024) → GELU → Linear(1024, P²·3)`.

In [ ]:
import torch.nn as nn

class PatchProbe(nn.Module):
    """Lightweight MLP decoder. Input: one token (C,). Output: one patch (3,P,P)."""
    def __init__(self, in_dim, patch_size=14):
        super().__init__()
        self.P = patch_size
        out_dim = 3 * patch_size * patch_size
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.GELU(),
            nn.Linear(1024, 1024),
            nn.GELU(),
            nn.Linear(1024, out_dim),
            nn.Sigmoid(),   # output in [0,1] to match unnorm pixel patches
        )

    def forward(self, token):            # token: (B, C)
        flat = self.net(token)           # (B, 3*P*P)
        return flat.view(-1, 3, self.P, self.P)

# Example: DINOv2 ViT-B has C=768
probe = PatchProbe(in_dim=768, patch_size=14).to(DEVICE)
dummy_token = torch.rand(4, 768).to(DEVICE)
print('Probe output shape:', probe(dummy_token).shape)   # expect (4, 3, 14, 14)

## 7 — Toy training loop (E1 MVP)

Train the probe on a tiny synthetic dataset to verify the pipeline end-to-end.  
Swap in a real DataLoader over ImageNet once you've confirmed shapes are correct.

In [ ]:
from torch.optim import Adam

def train_probe(probe, token_dataset, patch_dataset,
                n_epochs=5, lr=1e-3, batch_size=256):
    """
    token_dataset : (N, C) tensor of patch tokens
    patch_dataset : (N, 3, P, P) tensor of pixel patches (unnorm, [0,1])
    Returns list of per-epoch L1 losses.
    """
    from torch.utils.data import TensorDataset, DataLoader
    ds     = TensorDataset(token_dataset, patch_dataset)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    opt    = Adam(probe.parameters(), lr=lr)
    loss_fn = nn.L1Loss()
    history = []
    probe.train()
    for epoch in range(n_epochs):
        running = 0.0
        for tokens, patches in loader:
            tokens  = tokens.to(DEVICE)
            patches = patches.to(DEVICE)
            opt.zero_grad()
            pred = probe(tokens)
            loss = loss_fn(pred, patches)
            loss.backward()
            opt.step()
            running += loss.item()
        avg = running / len(loader)
        history.append(avg)
        print(f'  Epoch {epoch+1}/{n_epochs}  L1={avg:.4f}')
    probe.eval()
    return history

# ── Synthetic smoke-test (no ImageNet needed) ──────────────────────────────
N_FAKE = 512
C_DINO = 768
P      = 14

fake_tokens  = torch.randn(N_FAKE, C_DINO)
fake_patches = torch.rand(N_FAKE, 3, P, P)

dino_probe = PatchProbe(in_dim=C_DINO, patch_size=P).to(DEVICE)
history = train_probe(dino_probe, fake_tokens, fake_patches, n_epochs=3)
print('Smoke-test passed ✓')

## 8 — Evaluation metrics (PSNR, SSIM)

In [ ]:
import math
from skimage.metrics import structural_similarity as ssim_fn
import numpy as np

def psnr(pred, target):
    """pred, target: (N,3,P,P) tensors in [0,1]. Returns scalar."""
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return float('inf')
    return 10 * math.log10(1.0 / mse)

def batch_ssim(pred, target):
    """Average SSIM over a batch. pred/target: (N,3,P,P) in [0,1]."""
    pred_np   = pred.cpu().permute(0,2,3,1).numpy()    # (N,P,P,3)
    target_np = target.cpu().permute(0,2,3,1).numpy()
    scores = [
        ssim_fn(t, p, data_range=1.0, channel_axis=-1)
        for t, p in zip(target_np, pred_np)
    ]
    return float(np.mean(scores))

@torch.no_grad()
def eval_probe(probe, token_dataset, patch_dataset, batch_size=256):
    from torch.utils.data import TensorDataset, DataLoader
    loader = DataLoader(TensorDataset(token_dataset, patch_dataset),
                        batch_size=batch_size)
    all_psnr, all_ssim = [], []
    for tokens, patches in loader:
        pred = probe(tokens.to(DEVICE)).cpu().clamp(0, 1)
        all_psnr.append(psnr(pred, patches))
        all_ssim.append(batch_ssim(pred, patches))
    return {'PSNR': np.mean(all_psnr), 'SSIM': np.mean(all_ssim)}

# Smoke-test eval
metrics = eval_probe(dino_probe, fake_tokens, fake_patches)
print('Metrics on fake data (should be low — random tokens):', metrics)

## 9 — E1: Real token extraction from a single image (end-to-end check)

Use this to confirm the full pipeline (encoder → tokens → probe → metrics) before caching a big dataset.

In [ ]:
import urllib.request
from PIL import Image
import matplotlib.pyplot as plt

# Download a sample image
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
urllib.request.urlretrieve(url, 'sample.png')
img_pil = Image.open('sample.png').convert('RGB')

# Normalised tensor for encoder
img_t = transform(img_pil).unsqueeze(0)          # (1,3,224,224)

# Unnormalised tensor for ground-truth patches
img_unnorm = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])(img_pil).unsqueeze(0)

# Extract DINOv2 tokens
dino_tokens = encode_dino(img_t)[-1]             # (1, 256, 768)
h = w = IMG_SIZE // PATCH_SIZE                   # 16

# Extract pixel patches
gt_patches = extract_pixel_patches(img_unnorm, PATCH_SIZE)  # (256, 3, 14, 14)

# Probe prediction (untrained — just checking shapes)
flat_tokens = dino_tokens.squeeze(0)             # (256, 768)
with torch.no_grad():
    pred_patches = dino_probe(flat_tokens.to(DEVICE)).cpu()  # (256, 3, 14, 14)

print('Token shape:', flat_tokens.shape)
print('GT patch shape:', gt_patches.shape)
print('Pred patch shape:', pred_patches.shape)

# Quick visual — first 8 patches side by side
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(gt_patches[i].permute(1,2,0).clamp(0,1))
    axes[0, i].axis('off')
    axes[0, i].set_title('GT' if i == 0 else '')
    axes[1, i].imshow(pred_patches[i].permute(1,2,0).clamp(0,1))
    axes[1, i].axis('off')
    axes[1, i].set_title('Pred' if i == 0 else '')
plt.suptitle('Top: ground-truth patches | Bottom: untrained probe predictions')
plt.tight_layout()
plt.show()

## 10 — E2: Layer-wise token extraction (DINOv2)

Hooking layers 3, 6, 9, 12 as specified in the paper.

In [ ]:
TARGET_LAYERS = [2, 5, 8, 11]   # 0-indexed → corresponds to layers 3,6,9,12 in 1-indexed notation

layer_tokens = encode_dino(img_t, layers=TARGET_LAYERS)  # dict {layer_idx: (1,256,768)}

for layer_idx, toks in layer_tokens.items():
    print(f'Layer {layer_idx+1:2d} tokens: {toks.shape}')

# To run E2 properly: train a separate probe per layer and compare PSNR/SSIM
# The loop below shows the structure:
print()
print('E2 probe-per-layer structure:')
for layer_idx in TARGET_LAYERS:
    in_dim = layer_tokens[layer_idx].shape[-1]
    probe_l = PatchProbe(in_dim=in_dim, patch_size=PATCH_SIZE).to(DEVICE)
    print(f'  Layer {layer_idx+1}: probe input dim = {in_dim}  (ready to train)')

## 11 — E3: AnyUp decomposition (Path A vs Path B sketch)

Path A decodes from the raw encoder token.  
Path B decodes from AnyUp's upsampled output, average-pooled back to one vector per patch.  
The gap Δ = qual_A − qual_B isolates AnyUp's linear-combination loss.

In [ ]:
# ── Path A: raw tokens (already have these from section 9) ──────────────────
tokens_A = dino_tokens.squeeze(0)        # (N, 768)

# ── AnyUp upsampling ─────────────────────────────────────────────────────────
# Reshape tokens to spatial grid for AnyUp
lr_grid = tokens_to_grid(dino_tokens, h=h, w=w)   # (1, 768, 16, 16)

with torch.no_grad():
    hr_features = upsample_features(img_t, lr_grid)   # (1, 768, 224, 224)

print('AnyUp output shape:', hr_features.shape)

# ── Path B: average-pool AnyUp features back to one vector per original patch
# Each patch covered (P x P) = (14 x 14) pixels; average-pool over those pixels.
pooled_B = F.avg_pool2d(hr_features, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)  # (1,768,16,16)
tokens_B = pooled_B.squeeze(0).permute(1, 2, 0).reshape(-1, 768).cpu()          # (N, 768)

print('Path A tokens shape:', tokens_A.shape)
print('Path B tokens shape:', tokens_B.shape)

# ── Train separate probes for Path A and Path B, then compute Δ ─────────────
# (With real data this would be the full ImageNet loop;
#  here we just verify shapes with the single image.)
probe_A = PatchProbe(in_dim=768, patch_size=PATCH_SIZE).to(DEVICE)
probe_B = PatchProbe(in_dim=768, patch_size=PATCH_SIZE).to(DEVICE)
print('Path A and B probes ready to train.')

## 12 — Feature caching helper (for cluster runs)

Cache tokens to disk so you can train probes without re-running the heavy encoders every time.

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader as DL

def cache_tokens(encoder_fn, encoder_name, patch_size, imagenet_root,
                 max_images=5000, batch_size=64, layers=None):
    """
    Runs encoder_fn over up to max_images from imagenet_root and saves:
      CACHE_DIR/{encoder_name}_tokens.pt     — (N_total, C) patch tokens (last layer)
      CACHE_DIR/{encoder_name}_patches.pt    — (N_total, 3, P, P) unnorm pixel patches

    If layers is not None (E2 only), saves per-layer files instead.
    """
    dataset = ImageFolder(imagenet_root, transform=transform)
    loader  = DL(dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    all_tokens  = []
    all_patches = []
    seen = 0

    unnorm_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
    raw_dataset = ImageFolder(imagenet_root, transform=unnorm_tf)
    raw_loader  = DL(raw_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    for (imgs_norm, _), (imgs_raw, _) in zip(loader, raw_loader):
        if seen >= max_images:
            break
        B = imgs_norm.shape[0]
        token_dict = encoder_fn(imgs_norm, layers=layers) if layers else encoder_fn(imgs_norm)
        patches    = extract_pixel_patches(imgs_raw, patch_size)  # (B*N,3,P,P)

        # Last layer only for the simple cache
        last_key = max(token_dict.keys())
        toks = token_dict[last_key].reshape(-1, token_dict[last_key].shape[-1])  # (B*N,C)
        all_tokens.append(toks.cpu())
        all_patches.append(patches.cpu())
        seen += B
        print(f'  Cached {seen}/{max_images} images', end='\r')

    all_tokens  = torch.cat(all_tokens,  dim=0)
    all_patches = torch.cat(all_patches, dim=0)
    torch.save(all_tokens,  CACHE_DIR / f'{encoder_name}_tokens.pt')
    torch.save(all_patches, CACHE_DIR / f'{encoder_name}_patches.pt')
    print(f'\nSaved {all_tokens.shape[0]} patch-token pairs to {CACHE_DIR}')
    return all_tokens, all_patches

# ── To run on the cluster (uncomment):
# tokens, patches = cache_tokens(
#     encoder_fn=encode_dino,
#     encoder_name='dinov2_vitb14',
#     patch_size=PATCH_SIZE,
#     imagenet_root=IMAGENET_VAL,
#     max_images=5000,
# )
print('Caching helper ready ✓')

## 13 — Putting it all together: experiment runner

Once you have cached tokens, a full E1 run is just:

In [ ]:
def run_e1(encoder_name, in_dim, patch_size=14, n_epochs=10):
    """
    Load cached tokens, train probe, return metrics.
    Call this for each encoder after caching.
    """
    tokens  = torch.load(CACHE_DIR / f'{encoder_name}_tokens.pt')
    patches = torch.load(CACHE_DIR / f'{encoder_name}_patches.pt')

    # 80/20 train-val split
    N = tokens.shape[0]
    split = int(0.8 * N)
    tr_tok, va_tok   = tokens[:split],  tokens[split:]
    tr_pat, va_pat   = patches[:split], patches[split:]

    probe = PatchProbe(in_dim=in_dim, patch_size=patch_size).to(DEVICE)
    train_probe(probe, tr_tok, tr_pat, n_epochs=n_epochs)
    metrics = eval_probe(probe, va_tok, va_pat)
    print(f'[{encoder_name}] val metrics:', metrics)
    torch.save(probe.state_dict(), CACHE_DIR / f'{encoder_name}_probe.pt')
    return metrics

# Usage (after caching):
# results = {
#     'dinov2':  run_e1('dinov2_vitb14',  in_dim=768,  patch_size=14),
#     'clip':    run_e1('clip_vitb16',    in_dim=512,  patch_size=16),
#     'siglip':  run_e1('siglip_vitb16',  in_dim=768,  patch_size=16),
# }
print('Experiment runner ready ✓')

---

## Summary / next steps

| Who | Task |
|-----|------|
| Everyone this week | Run cells 0–11 locally, confirm shapes, try the single-image sanity check |
| Cluster person | Run `cache_tokens` for all three encoders on ImageNet val (5k images is enough to start) |
| Probe person | Swap in a real DataLoader and train `run_e1` for each encoder |
| AnyUp person | Run section 11 with real images; explore `vis_attn=True` to understand the upsampler |

**End-of-week meeting**: compare PSNR/SSIM across encoders (E1), look at layer curves (E2 hook structure is in section 10), and eyeball the Path A vs B delta (E3 section 11). Split the full experiments from there.